# House Stage 1 regression: no generic ballot, rolling 2018–2022

District margins forecast **without the generic ballot**: from `partisan_lean`, incumbency, and a small, theory-driven set of fundamentals, in the same expanding-window backtest as the other notebooks:

| eval year | train years |
|---|---|
| 2018 | 2006–2016 |
| 2020 | 2006–2018 |
| 2022 | 2006–2020 |

**No generic-ballot data enters any no-GB model**, in fitting or in prediction. Their formulas and their `require_complete` rows use no `gen_margin_raw` or `gb_*` column, which the guard cell below asserts. The ex-ante competitiveness filter (|`partisan_lean` + `gb_margin_2p_norm`| ≤ 10) uses the generic ballot, but only to choose which districts are scored, never in a model. The ex-post filter (|margin| ≤ 7.5) is unchanged. 2024 is never used (`USE_HOLDOUT = False`).

**Incumbency:** `inc_signed` unrestricted (OLS), or fixed at 2.5 or 5.0 (`run_ols_fixed`: OLS of the target minus `c · inc_signed`, with it added back to the predictions), each with and without `partisan_lean:inc_signed`.

**Additional predictors,** each added on its own to every base:

- **ACS:** `rel_state_college` (district − state college share, points) and `hispanic`, tested separately.
- **FEC:** `fundraising_advantage`. This is exploratory: historical totals run through December 31, after the election.
- **Economy,** from September of the election year, the last month published before Election Day:
  - `unemp_change_12m` (state, 12-month change)
  - `unemp_state_minus_us` (state − national rate)
  - `cpi_us_yoy` (national inflation)
  - `cpi_region_minus_us_yoy` (regional − national inflation)
  - `log_gas_yoy` (regional gasoline, 12-month log change)

  Each enters signed by the president's party (`pres_party_signed:<variable>`, +1 for a Democratic president, −1 for a Republican, known before every election), so its β is the effect on the president's party's margin. Without the generic ballot, these signed terms are the only way the national environment can enter these models.

**Two economic combinations,** and no more:

- **National economy:** the state's 12-month unemployment change + national inflation, the two halves of a misery-index judgment of the president.
- **Local economy:** unemployment and inflation relative to the nation.

Gasoline is not combined with CPI, since gasoline prices are part of the CPI. The ACS and FEC terms are not combined with the economy, which keeps the models from becoming kitchen sinks.

**Comparison with the generic-ballot finalists:** the same 11 feature sets on `GB_FORMULA` (with the generic ballot) with incumbency fixed at 2.5, with and without the interaction. `gb_inc_fixed_2.5_lean_x_inc`'s base is the finalists notebook's base model. Every model, with or without the generic ballot, scores exactly the same rows.

**Caveats:**

- **National-only variables:** `cpi_us_yoy` is the same for every district in a year, so its effect is estimated from 6–8 training years. 2022's 8.2% is far above every training year (1.1–4.9%), so it extrapolates.
- **Optimistic picks:** any feature set picked from these three eval years scores optimistically.

**Priority:** ex-ante MAE and winner accuracy first. ALL and ex-post metrics are diagnostics.

In [1]:
%load_ext autoreload
%autoreload 2
import sys
import warnings
from pathlib import Path

import pandas as pd
from IPython.display import Markdown

# The repository root is the first folder up from here that holds pyproject.toml, so this works at any depth
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT))  # repository root, for experiments.house
import house.regression as hr  # hr.<Tab> lists everything; names are in src/house/regression/__init__.py
from experiments.house import evaluation, tracking
from experiments.house.run import run_ols_fixed

In [2]:
import warnings

with warnings.catch_warnings(action="ignore"):  # loading warns with the same summaries printed below
    predictors = hr.load_predictors()  # data/processed/house/modeling.csv, every year (2006-2026)
    targets = hr.load_targets()        # data/processed/house/targets.csv, every year (2006-2024)
print(hr.validate_schema(predictors).summary())  # .issues lists every affected key

In [3]:
TARGET = "adj_margin_2p_norm"
NO_GB_FORMULA = f"{TARGET} ~ partisan_lean + inc_signed"              # no generic ballot
GB_FORMULA = f"{TARGET} ~ gen_margin_raw + partisan_lean + inc_signed"  # the classic formula, for comparison
ROLLING_WINDOWS = {  # eval year: its expanding training window
    2018: [2006, 2008, 2010, 2012, 2014, 2016],
    2020: [2006, 2008, 2010, 2012, 2014, 2016, 2018],
    2022: [2006, 2008, 2010, 2012, 2014, 2016, 2018, 2020],
}
USE_HOLDOUT = False     # never True here: no window touches HOLDOUT_YEARS (2024)
# Evaluation only (training is unaffected): observed, non-imputed targets of these race types are scored ("valid")
EVAL_RACE_TYPES = ["dr", "dr_multi", "dr_jungle", "dr_rcv", "dr_writein"]
# Competitive subsets of the valid targets: name -> (column, threshold); a row is in one when abs(column) <= threshold.
# column is a column name or arithmetic of them, from the targets or the predictors
EX_ANTE_THRESHOLD = 10   # points: any non-negative int or float, e.g. 5, 7, or 7.5
EX_POST_THRESHOLD = 7.5
COMPETITIVE = {
    # expected margin under uniform swing: two-party lean (2 x cook_pvi) + eve-of-election two-party generic ballot
    "competitive_ex_ante": ("partisan_lean + gb_margin_2p_norm", EX_ANTE_THRESHOLD),
    "competitive_ex_post": ("margin_2p_norm", EX_POST_THRESHOLD),  # observed two-party margin
}
SUMMARY_SECTIONS = {"ALL": "valid", "EX_ANTE_COMPETITIVE": "competitive_ex_ante",
                    "EX_POST_COMPETITIVE": "competitive_ex_post"}
RANK_BY = "mae"     # variants ranked by mean out-of-sample MAE ...
TIEBREAK = "rmse"   # ... ties broken by RMSE

## Models

Each block is one incumbency treatment on one base formula, and holds the same 11 feature sets (`FEATURE_SETS`). Its first model, `base`, is the base of every Δ in the block. For unrestricted OLS, `fixed={}`: `run_ols_fixed` with nothing fixed is `run_ols`.

In [4]:
FIXED_TERM = "inc_signed"
LEAN_X_INC = " + partisan_lean:inc_signed"

# The party holding the White House at each election, known before it: +1 Democratic, -1 Republican
PRES_PARTY_SIGNED = {2006: -1, 2008: -1, 2010: 1, 2012: 1, 2014: 1, 2016: 1, 2018: -1, 2020: -1, 2022: 1, 2024: 1,
                     2026: -1}
predictors = predictors.assign(pres_party_signed=predictors["year"].map(PRES_PARTY_SIGNED))

FEATURE_SETS = {  # model: the terms it adds to its block's base formula
    "base": [],
    "plus_rel_state_college": ["rel_state_college"],
    "plus_hispanic": ["hispanic"],
    "plus_fundraising": ["fundraising_advantage"],  # exploratory: post-election money in historical totals
    "pres_x_unemp_change_12m": ["pres_party_signed:unemp_change_12m"],
    "pres_x_unemp_state_minus_us": ["pres_party_signed:unemp_state_minus_us"],
    "pres_x_cpi_us_yoy": ["pres_party_signed:cpi_us_yoy"],
    "pres_x_cpi_region_minus_us_yoy": ["pres_party_signed:cpi_region_minus_us_yoy"],
    "pres_x_log_gas_yoy": ["pres_party_signed:log_gas_yoy"],
    # unemployment change + inflation, nationally judged
    "national_economy": ["pres_party_signed:unemp_change_12m", "pres_party_signed:cpi_us_yoy"],
    # unemployment and inflation relative to the nation
    "local_economy": ["pres_party_signed:unemp_state_minus_us", "pres_party_signed:cpi_region_minus_us_yoy"],
}
NO_GB_INCUMBENCY = {"ols": None, "inc_fixed_2.5": 2.5, "inc_fixed_5": 5.0}  # treatment: fixed coefficient (None: OLS)
GB_INCUMBENCY = {"inc_fixed_2.5": 2.5}  # the generic-ballot comparison: the finalists' fixed coefficient

BLOCKS = {  # block: (base formula, fixed inc_signed coefficient or None)
    **{f"no_gb_{name}{suffix}": (NO_GB_FORMULA + interaction, c)
       for name, c in NO_GB_INCUMBENCY.items() for suffix, interaction in [("", ""), ("_lean_x_inc", LEAN_X_INC)]},
    **{f"gb_{name}{suffix}": (GB_FORMULA + interaction, c)
       for name, c in GB_INCUMBENCY.items() for suffix, interaction in [("", ""), ("_lean_x_inc", LEAN_X_INC)]},
}
GB_FINALIST = ("gb_inc_fixed_2.5_lean_x_inc", "base")  # the generic-ballot finalist base: the reference below


def with_terms(base_formula, terms):
    return base_formula + "".join(f" + {term}" for term in terms)


pd.Series({block: base for block, (base, _) in BLOCKS.items()}, name="base formula").to_frame().assign(
    fixed_inc=[c for _, c in BLOCKS.values()])

,base formula,fixed_inc
no_gb_ols,adj_margin_2p_norm ~ partisan_lean + inc_signed,NaN
no_gb_ols_lean_x_inc,adj_margin_2p_norm ~ partisan_lean + inc_signe...,NaN
no_gb_inc_fixed_2.5,adj_margin_2p_norm ~ partisan_lean + inc_signed,2.5
no_gb_inc_fixed_2.5_lean_x_inc,adj_margin_2p_norm ~ partisan_lean + inc_signe...,2.5
no_gb_inc_fixed_5,adj_margin_2p_norm ~ partisan_lean + inc_signed,5.0
no_gb_inc_fixed_5_lean_x_inc,adj_margin_2p_norm ~ partisan_lean + inc_signe...,5.0
gb_inc_fixed_2.5,adj_margin_2p_norm ~ gen_margin_raw + partisan...,2.5
gb_inc_fixed_2.5_lean_x_inc,adj_margin_2p_norm ~ gen_margin_raw + partisan...,2.5


In [5]:
# Guard: no generic-ballot column in any no-GB model's formula or rows; then each family's shared rows
GB_COLUMNS = ["gen_margin_raw", *(column for column in predictors.columns if column.startswith("gb_"))]
NO_GB_FORMULAS = [with_terms(base, terms) for block, (base, _) in BLOCKS.items() if block.startswith("no_gb_")
                  for terms in FEATURE_SETS.values()]
GB_FORMULAS = [with_terms(base, terms) for block, (base, _) in BLOCKS.items() if block.startswith("gb_")
               for terms in FEATURE_SETS.values()]
for formula in NO_GB_FORMULAS:
    _, variables = evaluation.formula_variables(formula, predictors.columns)
    assert not set(variables) & set(GB_COLUMNS), f"generic-ballot column in a no-GB model: {formula}"

SHARED_COLUMNS = {  # require_complete: every column any model of the family uses
    "no_gb": evaluation.shared_columns(formulas=NO_GB_FORMULAS, columns=predictors.columns),
    "gb": evaluation.shared_columns(formulas=GB_FORMULAS, columns=predictors.columns),
}
assert not set(SHARED_COLUMNS["no_gb"]) & set(GB_COLUMNS)
SHARED_COLUMNS

{'no_gb': ['partisan_lean',
  'inc_signed',
  'rel_state_college',
  'hispanic',
  'fundraising_advantage',
  'pres_party_signed',
  'unemp_change_12m',
  'unemp_state_minus_us',
  'cpi_us_yoy',
  'cpi_region_minus_us_yoy',
  'log_gas_yoy'],
 'gb': ['gen_margin_raw',
  'partisan_lean',
  'inc_signed',
  'rel_state_college',
  'hispanic',
  'fundraising_advantage',
  'pres_party_signed',
  'unemp_change_12m',
  'unemp_state_minus_us',
  'cpi_us_yoy',
  'cpi_region_minus_us_yoy',
  'log_gas_yoy']}

In [6]:
# Collinearity of the candidate terms, on the latest window's training rows (2006-2020): the combinations pair terms
# that aren't near-duplicates
train = predictors[predictors["year"].isin(ROLLING_WINDOWS[max(ROLLING_WINDOWS)])]
terms = pd.DataFrame({
    "partisan_lean": train["partisan_lean"],
    "inc_signed": train["inc_signed"],
    "rel_state_college": train["rel_state_college"],
    "hispanic": train["hispanic"],
    "fundraising_advantage": train["fundraising_advantage"],
    **{f"pres_x_{v}": train["pres_party_signed"] * train[v]
       for v in ["unemp_change_12m", "unemp_state_minus_us", "cpi_us_yoy", "cpi_region_minus_us_yoy", "log_gas_yoy"]},
})
terms.corr().round(2).style.background_gradient(cmap="RdBu_r", vmin=-1, vmax=1).format(precision=2)

,partisan_lean,inc_signed,rel_state_college,hispanic,fundraising_advantage,pres_x_unemp_change_12m,pres_x_unemp_state_minus_us,pres_x_cpi_us_yoy,pres_x_cpi_region_minus_us_yoy,pres_x_log_gas_yoy
partisan_lean,1.00,0.72,0.06,0.32,0.80,-0.06,-0.02,-0.01,-0.10,-0.03
inc_signed,0.72,1.00,-0.00,0.24,0.81,-0.09,-0.02,-0.05,-0.07,-0.03
rel_state_college,0.06,-0.00,1.00,-0.25,0.01,0.00,-0.00,0.00,0.00,0.00
hispanic,0.32,0.24,-0.25,1.00,0.27,-0.08,0.04,0.02,-0.10,-0.05
fundraising_advantage,0.80,0.81,0.01,0.27,1.00,-0.05,-0.03,-0.15,-0.08,-0.09
pres_x_unemp_change_12m,-0.06,-0.09,0.00,-0.08,-0.05,1.00,0.17,0.08,-0.02,-0.19
pres_x_unemp_state_minus_us,-0.02,-0.02,-0.00,0.04,-0.03,0.17,1.00,0.06,-0.07,0.02
pres_x_cpi_us_yoy,-0.01,-0.05,0.00,0.02,-0.15,0.08,0.06,1.00,0.01,0.41
pres_x_cpi_region_minus_us_yoy,-0.10,-0.07,0.00,-0.10,-0.08,-0.02,-0.07,0.01,1.00,0.15
pres_x_log_gas_yoy,-0.03,-0.03,0.00,-0.05,-0.09,-0.19,0.02,0.41,0.15,1.00


In [7]:
# Every window: every block's 11 models. The missing-predictor warnings (FEC has no data for a few winners) repeat for
# every window and model; each run's exclusions are in its .excluded and .counts
with warnings.catch_warnings(action="ignore"):
    rolling = {
        eval_year: {
            block: [
                run_ols_fixed(
                    formula=with_terms(base_formula, terms),
                    train_years=train_years,
                    eval_years=[eval_year],
                    predict_years=[],
                    fixed={} if fixed_inc is None else {FIXED_TERM: fixed_inc},
                    predictors=predictors,
                    targets=targets,
                    use_holdout=USE_HOLDOUT,
                    require_complete=SHARED_COLUMNS["no_gb" if block.startswith("no_gb_") else "gb"],
                    race_types=EVAL_RACE_TYPES,
                    competitive=COMPETITIVE,
                    name=name,
                    save=False,
                )
                for name, terms in FEATURE_SETS.items()
            ]
            for block, (base_formula, fixed_inc) in BLOCKS.items()
        }
        for eval_year, train_years in ROLLING_WINDOWS.items()
    }

# matched samples: in every window, every model (with and without the generic ballot) trains and scores the same rows
for eval_year, blocks in rolling.items():
    rows = {(run.counts["train_rows_hash"], run.counts["eval_rows_hash"]) for runs in blocks.values() for run in runs}
    assert len(rows) == 1, f"{eval_year}: models use different rows"
print(f"{sum(len(runs) for runs in rolling[max(rolling)].values())} models per window, all on the same rows")

88 models per window, all on the same rows


## Overview: every model by block

Each grid is a mean over 2018, 2020 and 2022, each weighted equally. Rows are the feature sets and columns are the blocks; the best value in each column is bold.

- **Ex-ante MAE and winner accuracy come first.** ALL and ex-post follow as diagnostics.
- **Against the generic-ballot finalist base:** the last grids compare every model with `GB_FINALIST`, the finalists notebook's base. All models score the same rows, so these differences are matched.

In [8]:
def mean_table(subset):
    """Every model's metrics on `subset`, averaged over the windows, indexed by (block, model)."""
    return tracking.rolling_best_of(rolling, subset=subset, rank_by=RANK_BY, top=None, tiebreak=TIEBREAK)[0]


MEANS = {section: mean_table(subset) for section, subset in SUMMARY_SECTIONS.items()}


def grid(section, metric):
    """`metric` as a feature set x block grid."""
    return MEANS[section][metric].unstack("block").reindex(index=list(FEATURE_SETS), columns=list(BLOCKS))


GRIDS = [  # (section, metric, lower is better), in priority order
    ("EX_ANTE_COMPETITIVE", "mae", True),
    ("EX_ANTE_COMPETITIVE", "accuracy_pct", False),
    ("ALL", "mae", True),
    ("ALL", "accuracy_pct", False),
    ("EX_POST_COMPETITIVE", "mae", True),
    ("EX_POST_COMPETITIVE", "accuracy_pct", False),
]
for section, metric, lower in GRIDS:
    table = grid(section, metric)
    display(Markdown(f"### {section}: mean {metric}"))
    display((table.style.highlight_min if lower else table.style.highlight_max)(axis=0, props="font-weight: bold")
            .format(precision=1 if metric.endswith("_pct") else 2))

### EX_ANTE_COMPETITIVE: mean mae

block,no_gb_ols,no_gb_ols_lean_x_inc,no_gb_inc_fixed_2.5,no_gb_inc_fixed_2.5_lean_x_inc,no_gb_inc_fixed_5,no_gb_inc_fixed_5_lean_x_inc,gb_inc_fixed_2.5,gb_inc_fixed_2.5_lean_x_inc
run,,,,,,,,
base,9.03,9.04,6.51,6.38,6.65,6.63,5.28,5.12
plus_rel_state_college,9.04,9.06,6.71,6.57,6.80,6.77,5.50,5.33
plus_hispanic,9.03,9.04,6.55,6.44,6.68,6.68,5.33,5.18
plus_fundraising,5.52,5.50,4.91,4.90,5.49,5.41,4.64,4.65
pres_x_unemp_change_12m,8.93,8.92,7.01,6.72,7.01,6.85,5.49,5.37
pres_x_unemp_state_minus_us,9.02,9.04,6.54,6.40,6.68,6.66,5.30,5.14
pres_x_cpi_us_yoy,8.96,8.90,7.95,7.28,7.75,7.32,6.11,5.61
pres_x_cpi_region_minus_us_yoy,8.99,9.01,6.49,6.36,6.63,6.60,5.28,5.11
pres_x_log_gas_yoy,8.49,8.47,6.51,6.11,6.47,6.24,5.42,5.01


### EX_ANTE_COMPETITIVE: mean accuracy_pct

block,no_gb_ols,no_gb_ols_lean_x_inc,no_gb_inc_fixed_2.5,no_gb_inc_fixed_2.5_lean_x_inc,no_gb_inc_fixed_5,no_gb_inc_fixed_5_lean_x_inc,gb_inc_fixed_2.5,gb_inc_fixed_2.5_lean_x_inc
run,,,,,,,,
base,81.3,81.6,75.9,79.3,80.9,82.7,80.4,84.4
plus_rel_state_college,81.3,81.3,75.3,78.9,80.5,82.7,77.6,83.5
plus_hispanic,81.3,81.6,75.6,78.6,80.5,82.0,79.4,84.2
plus_fundraising,81.9,82.4,84.3,84.3,83.0,81.9,84.7,84.7
pres_x_unemp_change_12m,83.5,83.8,72.3,76.3,76.8,80.1,79.8,83.7
pres_x_unemp_state_minus_us,81.6,81.6,75.6,79.3,81.3,82.4,80.0,84.1
pres_x_cpi_us_yoy,78.4,78.8,67.7,70.6,70.6,71.7,77.4,81.5
pres_x_cpi_region_minus_us_yoy,81.3,81.3,76.0,78.9,80.9,82.3,80.4,85.2
pres_x_log_gas_yoy,82.0,82.3,73.8,75.8,77.4,81.5,80.0,81.7


### ALL: mean mae

block,no_gb_ols,no_gb_ols_lean_x_inc,no_gb_inc_fixed_2.5,no_gb_inc_fixed_2.5_lean_x_inc,no_gb_inc_fixed_5,no_gb_inc_fixed_5_lean_x_inc,gb_inc_fixed_2.5,gb_inc_fixed_2.5_lean_x_inc
run,,,,,,,,
base,7.36,7.35,6.24,6.33,6.20,6.29,5.69,5.75
plus_rel_state_college,7.36,7.35,6.33,6.40,6.26,6.33,5.77,5.83
plus_hispanic,7.40,7.41,6.34,6.46,6.29,6.40,5.80,5.89
plus_fundraising,6.03,6.06,5.94,5.95,6.02,6.04,5.94,5.95
pres_x_unemp_change_12m,7.12,7.13,6.38,6.50,6.24,6.35,5.97,5.98
pres_x_unemp_state_minus_us,7.38,7.38,6.31,6.41,6.27,6.36,5.71,5.78
pres_x_cpi_us_yoy,8.78,8.79,8.23,8.29,8.08,8.16,6.51,6.55
pres_x_cpi_region_minus_us_yoy,7.35,7.35,6.25,6.33,6.21,6.29,5.68,5.74
pres_x_log_gas_yoy,6.92,6.94,6.04,6.18,5.93,6.05,5.61,5.72


... (2 outputs left out)

### EX_POST_COMPETITIVE: mean mae

block,no_gb_ols,no_gb_ols_lean_x_inc,no_gb_inc_fixed_2.5,no_gb_inc_fixed_2.5_lean_x_inc,no_gb_inc_fixed_5,no_gb_inc_fixed_5_lean_x_inc,gb_inc_fixed_2.5,gb_inc_fixed_2.5_lean_x_inc
run,,,,,,,,
base,11.34,11.35,7.74,7.73,8.20,8.25,6.43,6.44
plus_rel_state_college,11.35,11.35,7.98,7.96,8.39,8.42,6.70,6.69
plus_hispanic,11.37,11.38,7.91,7.93,8.32,8.41,6.64,6.66
plus_fundraising,6.43,6.39,5.29,5.25,6.39,6.28,5.02,5.03
pres_x_unemp_change_12m,10.57,10.55,8.27,7.97,8.37,8.22,6.65,6.71
pres_x_unemp_state_minus_us,11.36,11.37,7.82,7.80,8.27,8.33,6.47,6.48
pres_x_cpi_us_yoy,11.38,11.32,9.18,8.62,9.31,8.97,7.21,6.83
pres_x_cpi_region_minus_us_yoy,11.31,11.32,7.72,7.70,8.17,8.22,6.41,6.42
pres_x_log_gas_yoy,10.57,10.55,7.86,7.50,8.05,7.87,6.57,6.32


### EX_POST_COMPETITIVE: mean accuracy_pct

block,no_gb_ols,no_gb_ols_lean_x_inc,no_gb_inc_fixed_2.5,no_gb_inc_fixed_2.5_lean_x_inc,no_gb_inc_fixed_5,no_gb_inc_fixed_5_lean_x_inc,gb_inc_fixed_2.5,gb_inc_fixed_2.5_lean_x_inc
run,,,,,,,,
base,69.6,69.6,66.5,69.3,70.0,71.2,68.9,72.7
plus_rel_state_college,70.1,70.1,64.9,68.3,69.5,71.7,65.8,72.4
plus_hispanic,70.1,70.1,66.0,68.8,69.5,71.2,68.4,72.8
plus_fundraising,70.1,70.8,72.9,72.9,71.8,70.1,72.4,72.4
pres_x_unemp_change_12m,73.5,74.0,63.2,66.1,66.0,69.4,68.6,71.7
pres_x_unemp_state_minus_us,69.6,70.6,66.5,69.3,70.6,71.7,68.4,72.2
pres_x_cpi_us_yoy,67.4,68.0,62.7,65.8,62.9,64.4,67.0,70.0
pres_x_cpi_region_minus_us_yoy,69.6,70.1,66.5,68.9,70.0,71.7,68.9,73.9
pres_x_log_gas_yoy,71.6,71.6,62.5,64.8,65.8,69.7,70.2,70.4


In [9]:
# Every model against the generic-ballot finalist base (matched rows): negative ΔMAE and positive Δaccuracy are better
for section, metric, lower in GRIDS[:2]:
    delta = grid(section, metric) - MEANS[section].loc[GB_FINALIST, metric]
    display(Markdown(f"### {section}: Δ{metric} vs `{GB_FINALIST[0]}` / `{GB_FINALIST[1]}`"))
    display(delta.style.format("{:+.2f}" if lower else "{:+.1f}")
            .map(lambda v: "" if pd.isna(v) or v == 0 else
                 f"color: {tracking.GOOD if (v < 0) == lower else tracking.BAD}"))

### EX_ANTE_COMPETITIVE: Δmae vs `gb_inc_fixed_2.5_lean_x_inc` / `base`

block,no_gb_ols,no_gb_ols_lean_x_inc,no_gb_inc_fixed_2.5,no_gb_inc_fixed_2.5_lean_x_inc,no_gb_inc_fixed_5,no_gb_inc_fixed_5_lean_x_inc,gb_inc_fixed_2.5,gb_inc_fixed_2.5_lean_x_inc
run,,,,,,,,
base,+3.91,+3.93,+1.39,+1.26,+1.53,+1.51,+0.16,+0.00
plus_rel_state_college,+3.93,+3.94,+1.60,+1.46,+1.68,+1.65,+0.39,+0.21
plus_hispanic,+3.91,+3.92,+1.43,+1.32,+1.57,+1.56,+0.22,+0.07
plus_fundraising,+0.41,+0.38,-0.20,-0.22,+0.37,+0.30,-0.48,-0.46
pres_x_unemp_change_12m,+3.81,+3.80,+1.90,+1.60,+1.89,+1.74,+0.37,+0.26
pres_x_unemp_state_minus_us,+3.91,+3.92,+1.43,+1.29,+1.56,+1.54,+0.18,+0.03
pres_x_cpi_us_yoy,+3.84,+3.78,+2.84,+2.16,+2.63,+2.20,+0.99,+0.49
pres_x_cpi_region_minus_us_yoy,+3.88,+3.89,+1.37,+1.24,+1.51,+1.48,+0.16,-0.01
pres_x_log_gas_yoy,+3.37,+3.36,+1.40,+1.00,+1.36,+1.12,+0.31,-0.11


### EX_ANTE_COMPETITIVE: Δaccuracy_pct vs `gb_inc_fixed_2.5_lean_x_inc` / `base`

block,no_gb_ols,no_gb_ols_lean_x_inc,no_gb_inc_fixed_2.5,no_gb_inc_fixed_2.5_lean_x_inc,no_gb_inc_fixed_5,no_gb_inc_fixed_5_lean_x_inc,gb_inc_fixed_2.5,gb_inc_fixed_2.5_lean_x_inc
run,,,,,,,,
base,-3.1,-2.8,-8.5,-5.2,-3.5,-1.7,-4.1,+0.0
plus_rel_state_college,-3.1,-3.1,-9.1,-5.6,-3.9,-1.8,-6.8,-1.0
plus_hispanic,-3.1,-2.8,-8.8,-5.9,-3.9,-2.5,-5.1,-0.3
plus_fundraising,-2.5,-2.1,-0.2,-0.2,-1.4,-2.5,+0.3,+0.3
pres_x_unemp_change_12m,-0.9,-0.6,-12.1,-8.1,-7.6,-4.4,-4.7,-0.7
pres_x_unemp_state_minus_us,-2.8,-2.8,-8.9,-5.2,-3.2,-2.1,-4.4,-0.4
pres_x_cpi_us_yoy,-6.1,-5.7,-16.7,-13.9,-13.8,-12.7,-7.0,-3.0
pres_x_cpi_region_minus_us_yoy,-3.1,-3.1,-8.4,-5.5,-3.5,-2.1,-4.1,+0.7
pres_x_log_gas_yoy,-2.5,-2.1,-10.6,-8.6,-7.0,-3.0,-4.5,-2.7


## Detail by section

Each block's base and its `DETAIL_TOP` best models by mean MAE (ties: RMSE), with every metric, the per-window MAE, and Δ against the block's base. `None` shows every model.

In [10]:
DETAIL_TOP = 3

for section, subset in SUMMARY_SECTIONS.items():
    mean, _ = tracking.rolling_best_of(rolling, subset=subset, rank_by=RANK_BY, top=DETAIL_TOP, tiebreak=TIEBREAK)
    display(Markdown(f"### {section} (`{subset}`): mean of {', '.join(map(str, ROLLING_WINDOWS))}"))
    display(tracking.style_best_of(mean))

### ALL (`valid`): mean of 2018, 2020, 2022

### EX_ANTE_COMPETITIVE (`competitive_ex_ante`): mean of 2018, 2020, 2022

### EX_POST_COMPETITIVE (`competitive_ex_post`): mean of 2018, 2020, 2022

## Coefficients

Averaged over the windows. The first table is each block's base. The second shows the coefficients of the terms each feature set adds. For a fixed block, `inc_signed` is the imposed value. A `pres_party_signed:<variable>` coefficient is the effect on the president's party's margin per unit of the variable.

In [11]:
_, COEFFICIENTS = tracking.rolling_best_of(rolling, rank_by=RANK_BY, top=None, tiebreak=TIEBREAK)
COEFFICIENTS.xs("base", level="run").round(3)

,Intercept,partisan_lean,inc_signed,rel_state_college,hispanic,fundraising_advantage,pres_party_signed:unemp_change_12m,pres_party_signed:unemp_state_minus_us,pres_party_signed:cpi_us_yoy,pres_party_signed:cpi_region_minus_us_yoy,pres_party_signed:log_gas_yoy,partisan_lean:inc_signed,gen_margin_raw
block,,,,,,,,,,,,,
no_gb_ols,3.001,0.816,12.208,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
no_gb_ols_lean_x_inc,3.186,0.820,12.138,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.010,NaN
no_gb_inc_fixed_2.5,2.757,1.037,2.500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
no_gb_inc_fixed_2.5_lean_x_inc,4.617,1.059,2.500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.100,NaN
no_gb_inc_fixed_5,2.819,0.980,5.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
no_gb_inc_fixed_5_lean_x_inc,4.245,0.997,5.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.077,NaN
gb_inc_fixed_2.5,-0.215,1.044,2.500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.791
gb_inc_fixed_2.5_lean_x_inc,1.732,1.068,2.500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.106,0.798


In [12]:
# the added terms' coefficients: feature set x block
added = pd.DataFrame({
    block: {name: ", ".join(f"{term} {COEFFICIENTS.loc[(block, name), term]:+.3f}" for term in terms)
            for name, terms in FEATURE_SETS.items() if terms}
    for block in BLOCKS
})
added.style.set_properties(**{"white-space": "pre-wrap", "text-align": "left"})

,no_gb_ols,no_gb_ols_lean_x_inc,no_gb_inc_fixed_2.5,no_gb_inc_fixed_2.5_lean_x_inc,no_gb_inc_fixed_5,no_gb_inc_fixed_5_lean_x_inc,gb_inc_fixed_2.5,gb_inc_fixed_2.5_lean_x_inc
plus_rel_state_college,rel_state_college -0.012,rel_state_college -0.013,rel_state_college -0.083,rel_state_college -0.083,rel_state_college -0.065,rel_state_college -0.065,rel_state_college -0.090,rel_state_college -0.091
plus_hispanic,hispanic +0.025,hispanic +0.026,hispanic +0.038,hispanic +0.043,hispanic +0.035,hispanic +0.039,hispanic +0.043,hispanic +0.049
plus_fundraising,fundraising_advantage +4.420,fundraising_advantage +4.445,fundraising_advantage +4.942,fundraising_advantage +4.965,fundraising_advantage +4.424,fundraising_advantage +4.474,fundraising_advantage +4.606,fundraising_advantage +4.603
pres_x_unemp_change_12m,pres_party_signed:unemp_change_12m +0.755,pres_party_signed:unemp_change_12m +0.756,pres_party_signed:unemp_change_12m +0.646,pres_party_signed:unemp_change_12m +0.658,pres_party_signed:unemp_change_12m +0.676,pres_party_signed:unemp_change_12m +0.685,pres_party_signed:unemp_change_12m -0.160,pres_party_signed:unemp_change_12m -0.151
pres_x_unemp_state_minus_us,pres_party_signed:unemp_state_minus_us -0.175,pres_party_signed:unemp_state_minus_us -0.178,pres_party_signed:unemp_state_minus_us -0.245,pres_party_signed:unemp_state_minus_us -0.272,pres_party_signed:unemp_state_minus_us -0.227,pres_party_signed:unemp_state_minus_us -0.248,pres_party_signed:unemp_state_minus_us -0.065,pres_party_signed:unemp_state_minus_us -0.091
pres_x_cpi_us_yoy,pres_party_signed:cpi_us_yoy -1.787,pres_party_signed:cpi_us_yoy -1.788,pres_party_signed:cpi_us_yoy -1.913,pres_party_signed:cpi_us_yoy -1.903,pres_party_signed:cpi_us_yoy -1.880,pres_party_signed:cpi_us_yoy -1.872,pres_party_signed:cpi_us_yoy -0.826,pres_party_signed:cpi_us_yoy -0.776
pres_x_cpi_region_minus_us_yoy,pres_party_signed:cpi_region_minus_us_yoy +0.481,pres_party_signed:cpi_region_minus_us_yoy +0.487,pres_party_signed:cpi_region_minus_us_yoy +0.639,pres_party_signed:cpi_region_minus_us_yoy +0.676,pres_party_signed:cpi_region_minus_us_yoy +0.600,pres_party_signed:cpi_region_minus_us_yoy +0.627,pres_party_signed:cpi_region_minus_us_yoy +0.520,pres_party_signed:cpi_region_minus_us_yoy +0.560
pres_x_log_gas_yoy,pres_party_signed:log_gas_yoy -0.116,pres_party_signed:log_gas_yoy -0.116,pres_party_signed:log_gas_yoy -0.136,pres_party_signed:log_gas_yoy -0.136,pres_party_signed:log_gas_yoy -0.131,pres_party_signed:log_gas_yoy -0.131,pres_party_signed:log_gas_yoy -0.104,pres_party_signed:log_gas_yoy -0.104
national_economy,"pres_party_signed:unemp_change_12m +0.877, pres_party_signed:cpi_us_yoy -1.807","pres_party_signed:unemp_change_12m +0.877, pres_party_signed:cpi_us_yoy -1.808","pres_party_signed:unemp_change_12m +0.783, pres_party_signed:cpi_us_yoy -1.930","pres_party_signed:unemp_change_12m +0.791, pres_party_signed:cpi_us_yoy -1.919","pres_party_signed:unemp_change_12m +0.810, pres_party_signed:cpi_us_yoy -1.898","pres_party_signed:unemp_change_12m +0.817, pres_party_signed:cpi_us_yoy -1.889","pres_party_signed:unemp_change_12m +0.125, pres_party_signed:cpi_us_yoy -0.838","pres_party_signed:unemp_change_12m +0.113, pres_party_signed:cpi_us_yoy -0.787"
local_economy,"pres_party_signed:unemp_state_minus_us -0.160, pres_party_signed:cpi_region_minus_us_yoy +0.431","pres_party_signed:unemp_state_minus_us -0.163, pres_party_signed:cpi_region_minus_us_yoy +0.435","pres_party_signed:unemp_state_minus_us -0.223, pres_party_signed:cpi_region_minus_us_yoy +0.570","pres_party_signed:unemp_state_minus_us -0.250, pres_party_signed:cpi_region_minus_us_yoy +0.599","pres_party_signed:unemp_state_minus_us -0.208, pres_party_signed:cpi_region_minus_us_yoy +0.535","pres_party_signed:unemp_state_minus_us -0.228, pres_party_signed:cpi_region_minus_us_yoy +0.557","pres_party_signed:unemp_state_minus_us -0.046, pres_party_signed:cpi_region_minus_us_yoy +0.498","pres_party_signed:unemp_state_minus_us -0.072